# RAG 기본 프로세스

In [3]:
try:
    import google.colab
    inColab = True
except ImportError:
    inColab = False

### 라이브러리 설치

In [4]:
# !pip install -q ollama==0.6.1
# !pip install -q langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6
# !pip install -q chromadb==1.3.7 pypdf==6.4.2 sentence-transformers==5.2.0
# !pip install -q openai==2.12.0

In [5]:

# langchain, langchain-community, langchain-openai: RAG 구성의 핵심
# chromadb: 벡터 데이터베이스
# pypdf: PDF 로딩
# sentence-transformers: 한국어 임베딩
# openai: GPT/임베딩 연동
# llama-cpp-python, ollama: 로컬/서버형 모델
# ★ llama-cpp-python 및 ollama는 필요 시

# !pip install llama-cpp-python==0.3.16



### 라이브러리 선언

In [6]:
# langchain, langchain-community, langchain-openai: RAG 구성핵심
# chromadb: 벡터DB pypdf: PDF sentence-transformers: 한국어 임베딩, ai: GPT연동 cpp/ollama:로컬

import os

# RAG 프로세스 핵심 라이브러리
import langchain, chromadb, pypdf
# PDF 로더
from langchain_community.document_loaders import PyPDFLoader
# 텍스트 청크 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 임베딩모델 OpenAI 라이브러리
from langchain_openai import OpenAIEmbeddings
# 임베딩모델 무료 라이브러리 (허깅페이스)
from langchain_community.embeddings import HuggingFaceEmbeddings
# 벡터 DB 생성 라이브러리
from langchain_community.vectorstores import Chroma
# 모델 GPT 사용 라이브러리
from langchain_openai import ChatOpenAI
# 모델 GGUF 사용 라이브러리
from langchain_community.llms import LlamaCpp
# 모델 OLLAMA 사용 라이브러리
from langchain_community.llms import Ollama
# 임베딩모델 Ollama 라이브러리
from langchain_community.embeddings import OllamaEmbeddings
# 프롬프트 템플릿 라이브러리
from langchain_core.prompts import PromptTemplate
# RAG 체인 구성 라이브러리
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

### 환경 설정 *** 본인 OPENAI_API키 로 설정 필요

In [7]:
# 폴더 구조 설정
DATA_DIR = "dataset"
PDF_PATH = os.path.join(DATA_DIR, "모집요강.pdf")
PERSIST_DIR = "vector_db"
MODEL_DIR = "models"
OPENAI_APIKEY = "OPENAI_API키"

# LLM 모델 선택: "gpt" 또는 "ollama"
MODEL_TYPE = "ollama"   # "gpt" 또는 "ollama" 중 선택
OLLAMA_MODEL = "gemma4:e2b"  # Ollama 사용 시 모델명

# 폴더 생성
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("PDF 경로:", PDF_PATH)
print("저장 경로:", PERSIST_DIR)

데이터 폴더: dataset
PDF 경로: dataset\모집요강.pdf
저장 경로: vector_db


In [8]:
# 설정값 정의
CHUNK_SIZE = 800        # 청크 크기 (권장: 600-1200)
CHUNK_OVERLAP = 150     # 청크 오버랩 (권장: 100-200)
K_RETRIEVAL = 4         # 검색할 문서 수 (권장: 3-6)

print("설정된 하이퍼파라미터:")
print(f"청크 크기: {CHUNK_SIZE}")
print(f"오버랩: {CHUNK_OVERLAP}")
print(f"검색 수: {K_RETRIEVAL}")

print("\n 설정값 영향:")
print("   - 큰 청크: 문맥 유지↑, 검색 정밀도↓")
print("   - 많은 오버랩: 문맥 단절 방지, 중복↑")
print("   - 높은 K값: 정확도↑, 속도/비용↑")

설정된 하이퍼파라미터:
청크 크기: 800
오버랩: 150
검색 수: 4

 설정값 영향:
   - 큰 청크: 문맥 유지↑, 검색 정밀도↓
   - 많은 오버랩: 문맥 단절 방지, 중복↑
   - 높은 K값: 정확도↑, 속도/비용↑


# 1. 데이터 준비

### 1-1. Data Load

In [9]:
# PDF 로딩 실행
# PDF 로더 생성 및 문서 로딩 (페이지 단위 로딩)
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f" {len(docs)}개 페이지 로딩 완료!")
print(f" 첫 번째 페이지 미리보기 (300자):")
print("-" * 50)
print(docs[0].page_content[:300])
print("-" * 50)
print(f" 메타데이터(page 및 source): {docs[0].metadata}")

 20개 페이지 로딩 완료!
 첫 번째 페이지 미리보기 (300자):
--------------------------------------------------
- 1 -
한국폴리텍대학 서울강서캠퍼스2026학년도 하이테크과정 모집요강   (주소) 서울특별시 강서구 우장산로10길 112 (대표번호) 02-2186-58001 모집학과(직종) 및 모집인원
 ※ 모집학과 및 모집 차수별 모집인원은 대학 사정에 따라 변경될 수 있음 ※ 기회균등 모집인원 미달시 일반선발 인원으로 모집(또한 각 전형 미달 발생 시 우선선발과 일반선발 교차선발) ※ 기회균등 대상자 (「직업교육훈련촉진법」시행령 제10조 해당자) ※ 정원의 110%까지 선발하며, 모집 차수별 미달 인원은 이월하여 다음 차수에 모집(모집1차
--------------------------------------------------
 메타데이터(page 및 source): {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2018 10.0.0.14515', 'creationdate': '2025-10-27T18:45:54+09:00', 'author': 'Moel', 'moddate': '2025-10-27T18:45:54+09:00', 'pdfversion': '1.4', 'source': 'dataset\\모집요강.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}


### 1-2. Split

In [10]:
# 텍스트 청크 분할 실행
print(" 텍스트 청크 분할 시작...")

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

# 문서 분할 실행
chunks = text_splitter.split_documents(docs)

print(f"청크 분할 결과:")
print(f"원본 페이지 수: {len(docs)}")
print(f"생성된 청크 수: {len(chunks)}")


print(f"\n첫 번째 청크 샘플:")
print("-" * 50)
print(chunks[0].page_content[:200] + "...")
print("-" * 50)
print(f"청크 메타데이터: {chunks[0].metadata}")

 텍스트 청크 분할 시작...
청크 분할 결과:
원본 페이지 수: 20
생성된 청크 수: 46

첫 번째 청크 샘플:
--------------------------------------------------
- 1 -
한국폴리텍대학 서울강서캠퍼스2026학년도 하이테크과정 모집요강   (주소) 서울특별시 강서구 우장산로10길 112 (대표번호) 02-2186-58001 모집학과(직종) 및 모집인원
 ※ 모집학과 및 모집 차수별 모집인원은 대학 사정에 따라 변경될 수 있음 ※ 기회균등 모집인원 미달시 일반선발 인원으로 모집(또한 각 전형 미달 발생 시 우선선발과 ...
--------------------------------------------------
청크 메타데이터: {'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2018 10.0.0.14515', 'creationdate': '2025-10-27T18:45:54+09:00', 'author': 'Moel', 'moddate': '2025-10-27T18:45:54+09:00', 'pdfversion': '1.4', 'source': 'dataset\\모집요강.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}


### 1-3. 임베딩 Store

In [ ]:
# 임베딩 모델 선택 및 초기화
# MODEL_TYPE에 따라 임베딩 자동 선택
# ollama: bge-m3 (Ollama 로컸), gpt: OpenAI, 그 외: HuggingFace 한국어
USE_OPENAI_EMBEDDING = True  # GPT 모드일 때 True, 개별 설정 원할 시 직접 변경

if MODEL_TYPE == "ollama":
    # Ollama 로컸 임베딩 (bge-m3)
    print("Ollama bge-m3 임베딩 모델 로딩...")
    embedding_model = OllamaEmbeddings(
        model="bge-m3"
    )
    print("Ollama bge-m3 임베딩 준비 완료")

elif USE_OPENAI_EMBEDDING:
    # OpenAI 임베딩 사용
    print("OpenAI 임베딩 모델 로딩...")
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY
    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
    print("OpenAI 유료 임베딩 준비 완료")

else:
    # 한국어 임베딩 사용
    print("한국어 임베딩 무료 모델 로딩...")
    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("무료 임베딩 모델 준비 완료")


한국어 임베딩 무료 모델 로딩...


C:\Users\hk\AppData\Local\Temp\ipykernel_10296\2540627272.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\hk\llm_allone\.venv4rag\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hk\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

### 벡터 데이터베이스의 역할
- 임베딩 벡터 저장/색인/검색**(KNN)
- 메타데이터와 함께 관리**(source, page)
- RAG에서 검색 단계**의 성능/속도 담당


```
[문서 임베딩] → [컬렉션 저장] → [인덱싱] → [쿼리 임베딩] → [최근접 이웃 검색] → [Top-K 결과]
```

In [ ]:
# ChromaDB 생성 및 벡터 저장

print("ChromaDB 벡터 저장소 생성...")
print(f"처리할 청크 수: {len(chunks)}")

# ChromaDB 생성 (임베딩과 함께)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR
)

# 영구 저장 (최신 버전에서는 자동으로 저장됨)
# vectorstore.persist()
print("ChromaDB 영구 저장 완료!")

# 검색기 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": K_RETRIEVAL}
)

# 결과 확인
collection_count = vectorstore._collection.count()
print(f"저장된 벡터 수: {collection_count}")
print(f"검색 설정: Top-{K_RETRIEVAL}")

# 테스트 검색 실행 (수정된 부분)
test_query = "면접 일정"

# 방법 1: invoke() 사용 (권장)
test_results = retriever.invoke(test_query)

# 방법 2: vectorstore에서 직접 검색
# test_results = vectorstore.similarity_search(test_query, k=K_RETRIEVAL)

print(f"\n테스트 검색 ('{test_query}'):")
print(f"검색된 문서 수: {len(test_results)}")
if test_results:
    print(f"   첫 번째 결과 (100자): {test_results[0].page_content[:100]}...")


ChromaDB 벡터 저장소 생성...
처리할 청크 수: 46
ChromaDB 영구 저장 완료!
저장된 벡터 수: 92
검색 설정: Top-4

테스트 검색 ('면접 일정'):
검색된 문서 수: 4
   첫 번째 결과 (100자): . 성적반영 요소 및 배점과정성적반영 요소성적 적용 비율면접(점수)하이테크과정100면접비율 100% 나. 세부사항 전형실시내용진행방식비 고면접고사․ NCS기반 종합면접․ 학업 및 ...


# 2. 정보 검색 및 텍스트 생성

### 2-1. 모델선언 및 Retrieval

In [ ]:
# LLM 모델 선택 및 초기화
# 모델 설정 (gpt/gguf/ollama 중 선택)
# 위 환경설정 셀에서 MODEL_TYPE 조정 가능 ("gpt" 또는 "ollama")
TEMPERATURE = 0.2     # 사실 기반 답변용 (0~1)

print(f"{MODEL_TYPE.upper()} 모델 초기화...")

if MODEL_TYPE == "gpt":
    # GPT 모델 사용
    print("GPT 모델 로딩...")
    # API 키 설정 (실제 사용시 입력 필요)
    os.environ["OPENAI_API_KEY"] = OPENAI_APIKEY

    llm = ChatOpenAI(
        model="gpt-3.5-turbo",  # 또는 "gpt-4o-mini"
        temperature=TEMPERATURE
    )
    print("GPT 모델 준비 완료")

elif MODEL_TYPE == "gguf":
    # ★ 수정포인트
    # 모델 경로 (실제 사용시 다운로드 필요)
    model_path = "/content/llama-model.gguf"

    llm = LlamaCpp(
        model_path=model_path,
        n_ctx=4096,
        n_threads=2,
        temperature=TEMPERATURE
    )
    print("GGUF 모델 준비 완료")

elif MODEL_TYPE == "ollama":
    # Ollama 모델 사용
    print("Ollama 모델 로딩...")
    llm = Ollama(
        model=OLLAMA_MODEL,
        temperature=TEMPERATURE
    )
    print("Ollama 모델 준비 완료")

print(f"설정된 모델: {MODEL_TYPE.upper()}")
print(f"Temperature: {TEMPERATURE}")

GPT 모델 초기화...
GPT 모델 로딩...
GPT 모델 준비 완료
설정된 모델: GPT
Temperature: 0.2


In [ ]:
# 테스트
try:
    test_response = llm.invoke("안녕하세요")

    # ChatModel (GPT)과 LLM의 반환 타입이 다름
    if MODEL_TYPE == "gpt":
        # ChatOpenAI는 객체를 return 하여 content
        response_text = test_response.content
    else:
        # LlamaCpp, Ollama는 문자열 반환
        response_text = test_response

    print(f"모델 테스트 성공: {response_text[:50]}...")

except Exception as e:
    print(f"모델 테스트 실패: {e}")
    print("API 키나 모델 경로를 확인하세요!")


모델 테스트 성공: 안녕하세요! 무엇을 도와드릴까요?...


### 2-2. RAG체인 구성 및 Generation

### RAG 체인: 질문 검색 생성, 프롬프트템플릿, 체인연결

```
[사용자 질문] → [쿼리 임베딩] → [Top-K 검색(Chroma)] → [컨텍스트 구성] → [LLM 응답 생성] → [답변 + 출처]
```

In [ ]:
# RAG용 프롬프트 템플릿 정의
PROMPT_TEMPLATE = """
다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
"""

# PromptTemplate 객체 생성
prompt_template = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("프롬프트 템플릿 생성 완료!")
print("\n템플릿 내용:")
print("-" * 50)
print(PROMPT_TEMPLATE[:200] + "...")
print("-" * 50)

# 템플릿 테스트
test_context = "HKCODE 유튜브 채널은 AI관련 정보공유"
test_question = "HKCODE 유튜브 채널은 누가 운영하나요?"

formatted_prompt = prompt_template.format(
    context=test_context,
    question=test_question
)

print("\n프롬프트 포맷팅 테스트:")
print(formatted_prompt)

프롬프트 템플릿 생성 완료!

템플릿 내용:
--------------------------------------------------

다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
{context}

질문: {question}
답변:
...
--------------------------------------------------

프롬프트 포맷팅 테스트:

다음 컨텍스트만을 사용해서 한국어로 간결하고 정확하게 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하고, 추측하지 마세요.
가능하면 출처 정보(페이지 번호)도 함께 제공하세요.

컨텍스트:
HKCODE 유튜브 채널은 AI관련 정보공유

질문: HKCODE 유튜브 채널은 누가 운영하나요?
답변:



In [ ]:
# 출처 문서 포함 RAG 체인
print("RAG 체인 구성 중 (출처 문서 반환)...")

# 문서 포맷팅 함수
def format_docs(docs):
    """검색된 문서를 텍스트로 변환"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[출처 {i}] 페이지 {page}\n{content}")
    return "\n\n".join(formatted)

# Step 1: 검색 및 문맥 준비
retrieval_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "source_documents": retriever
    }
)

# Step 2: 답변 생성 체인
answer_chain = (
    {
        "context": lambda x: x["context"],
        "question": lambda x: x["question"]
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# Step 3: 전체 체인 결합
rag_chain = retrieval_chain | RunnablePassthrough.assign(answer=answer_chain)

print("RAG 체인 생성 완료!")

print("\n체인 구성요소:")
print(f"LLM: {MODEL_TYPE.upper()}")
print(f"Retriever: ChromaDB (K={K_RETRIEVAL})")
print(f"Prompt: 커스텀 템플릿")
print(f"Source Docs: 반환 활성화")


RAG 체인 구성 중 (출처 문서 반환)...
RAG 체인 생성 완료!

체인 구성요소:
LLM: GPT
Retriever: ChromaDB (K=4)
Prompt: 커스텀 템플릿
Source Docs: 반환 활성화


### 질의 테스트

In [ ]:
# 질문
question = "스마트금융과 모집 일정은?"

print(f"\n질문: {question}")

# 전체 결과 받기
full_result = rag_chain.invoke(question)

# 각 요소 분리 저장
answer = full_result['answer']              # 답변만
source_docs = full_result['source_documents']  # 출처 문서 리스트
context = full_result['context']            # 포맷된 컨텍스트
question_echo = full_result['question']     # 질문 (확인용)

# 출력
print(f"\n답변:\n{answer}")

print(f"\n참조 문서 ({len(source_docs)}개):")
for i, doc in enumerate(source_docs, 1):
    page = doc.metadata.get('page', '?')
    source = doc.metadata.get('source', '알 수 없음')
    content_preview = doc.page_content[:80].replace('\n', ' ')
    print(f"   [{i}] 페이지 {page}")
    print(f"       {content_preview}...")


질문: 스마트금융과 모집 일정은?

답변:
스마트금융과의 1차 모집 일정은 2025.11.03.(월) 10시부터 2025.12.05.(금) 17시까지이며, 2차 모집 일정은 2025.12.05.(금) 18시부터 2026.01.29.(목) 17시까지입니다. (출처 1, 페이지 0)

참조 문서 (4개):
   [1] 페이지 0
       ■ 면접 안내 - 개인별 면접 시간 및 장소는 면접일 전일 캠퍼스    홈페이지에 공지되며, 개별 공지하지 않음 - 면접 불참자는 불합격 처리됨...
   [2] 페이지 0
       ■ 면접 안내 - 개인별 면접 시간 및 장소는 면접일 전일 캠퍼스    홈페이지에 공지되며, 개별 공지하지 않음 - 면접 불참자는 불합격 처리됨...
   [3] 페이지 14
       - 15 - 서식 4 (하이테크과정) (비정규형태근로자) 근무 확인서근무 확인서성    명 : 생년월일 : 근무 기간 : 0000. 00. 00...
   [4] 페이지 14
       - 15 - 서식 4 (하이테크과정) (비정규형태근로자) 근무 확인서근무 확인서성    명 : 생년월일 : 근무 기간 : 0000. 00. 00...


### 참고. FAST API 연동

In [ ]:
if inColab == True:
    !pip install -U nest-asyncio==1.6.0 pyngrok==7.2.4 uvicorn==0.34.2 fastapi==0.115.12
import uvicorn
# 서버 관리용 fastapi 의존 라이브러리
import uvicorn

# fast api 라이브러리
from fastapi import FastAPI

# 데이터프레임 및 수 처리 라이브러리
import pandas as pd
# 인터페이스 데이터 관리를 위한 라이브러리
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.38.0
    Uninstalling uvicorn-0.38.0:
      Successfully uninstalled uvicorn-0.38.0
  Attempting uninstall: starlette
    Found existing installation: starlette 0.50.0
    Uninstalling starlette-0.50.0:
      Successfully uninstalled starlette-0.50.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.123.10
    Uninstalling fastapi-0.123.10:
      Successfully uninstalled fastapi-0.123.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sse-starlette 3.0.4 requires starlette>=0.49.1, but you have starlette 0.46.2 which is incompatible.

In [ ]:
origins = ["*"]

app = FastAPI(title="ML API")

# CORS 미들웨어 추가
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 모든 origin 허용
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

In [ ]:
class InDataset(BaseModel):
    question : str

In [ ]:
# # 테스트 코드
# # print(x)
# question = "스마트금융과 모집 일정은?"
# response = rag_chain.invoke(question)
# response1 = response["answer"]
# response2 = response["source_documents"]
# print(response1)
# print(response2)

In [ ]:
@app.post("/predict", status_code=200)
async def predict_tf(x: InDataset):
    print(x)
    question = "스마트금융과 모집 일정은?"
    response = rag_chain.invoke(question)
    response1 = response["answer"]
    response2 = response["source_documents"]
    print(response1)
    print(response2)

    return {"prediction": response1,
            "references": response2 }

@app.get('/')
async def root():
    return {"message": "online"}

In [ ]:
if inColab == True:
    import nest_asyncio
    from pyngrok import ngrok
    import uvicorn

    auth_token = "371Ga2zptlxD4G8mq2oY7exyIUm_363aLB8Dvt6v6VqAMkzFz"
    ngrok.set_auth_token(auth_token)
    ngrokTunnel = ngrok.connect(8000)
    print("공용 URL", ngrokTunnel.public_url)
    nest_asyncio.apply()
    uvicorn.run(app, port=8000)

else:
    import nest_asyncio
    nest_asyncio.apply()

    # Uvicorn 실행
    if __name__ == "__main__":
        uvicorn.run(app, host="0.0.0.0", port=9999, log_level="debug")

공용 URL https://45a45b99130d.ngrok-free.app


INFO:     Started server process [228]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


question='스마트금융과 모집일정은?'
스마트금융과의 1차 모집 일정은 2025년 11월 3일(월)부터 12월 5일(금)까지이며, 2차 모집 일정은 2025년 12월 5일(금)부터 2026년 1월 29일(목)까지입니다. (출처 1, 페이지 0)
[Document(metadata={'moddate': '2025-10-27T18:45:54+09:00', 'pdfversion': '1.4', 'source': 'datasets/모집요강.pdf', 'page_label': '1', 'total_pages': 20, 'creator': 'Hwp 2018 10.0.0.14515', 'page': 0, 'author': 'Moel', 'creationdate': '2025-10-27T18:45:54+09:00', 'producer': 'Hancom PDF 1.3.0.550'}, page_content='■ 면접 안내 - 개인별 면접 시간 및 장소는 면접일 전일 캠퍼스    홈페이지에 공지되며, 개별 공지하지 않음 - 면접 불참자는 불합격 처리됨 ※ 캠퍼스 사정에 따라 면접일을 연장하여 실시할 수 있음■ 합격 자: 홈 페이 지  발표(본 인 직 접  확인, 개 별  공지 하 지 않 음)■ 지원자별 해당 제출서류 주소지(등기우편) - (07684) 서울특별시 강서구 우장산로10길 112   한국폴리텍대학 서울강서캠퍼스 교학처   (하이테크과정 입시담당자 앞) ※ 입학전형은 대학 사정에 따라 입학전형관리소위원회의 심의를 통하여 변경될 수 있음\n과 정학 과직 종모집정원모집인원모집 1차모집 2차소계일반균형소계일반균형하이테크(주간1년) 총 계 8088402911483513사이버보안사이버보안202210731293스마트금융핀테크202210731293출판편집디자인출판40442015524177\n과정모집차수인터넷 원서접수면접합격자발표 및 등록하이테크1차2025.11.03.(월) 10시~2025.12.05.(금) 17시2025.12.11.(목)2025.12.19.(금) 10시~2025

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [228]


### 참고. 대화형 모드

In [ ]:
# 대화형 RAG 체인 생성
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("대화형 RAG 체인 설정...")

# 대화형 프롬프트 템플릿
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질문답변 AI입니다.
다음 컨텍스트와 대화 기록을 참고하여 한국어로 답변하세요.
컨텍스트에 없는 내용은 모른다고 답변하세요."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """컨텍스트:
{context}

질문: {question}""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[문서 {i} - 페이지 {page}]\n{content}")
    return "\n\n".join(formatted)

# 대화 기록 저장소 (간단한 리스트)
chat_history = []

# 대화형 RAG 체인
def chat_rag(question):
    """대화 기록을 유지하며 답변을 생성하는 함수"""

    # 검색
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # 답변 생성
    response = chat_prompt | llm | StrOutputParser()
    answer = response.invoke({
        "context": context,
        "question": question,
        "chat_history": chat_history
    })

    # 대화 기록에 추가
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return {
        "question": question,
        "answer": answer,
        "source_documents": docs
    }

print("대화형 RAG 체인 준비 완료!")

# 대화 예시 테스트
print("\n대화형 테스트:")
print("="*50)

# 첫 번째 질문
q1 = "지원 자격이 뭐야?"
print(f"질문 1: {q1}")
result1 = chat_rag(q1)
print(f"답변 1: {result1['answer']}")

# 두 번째 연관 질문
q2 = "그럼 제출 서류는 뭐가 필요해?"
print(f"\n질문 2: {q2}")
result2 = chat_rag(q2)
print(f"답변 2: {result2['answer']}")

# 세 번째 질문 (이전 대화 참조)
q3 = "그거 어디에 제출하면 돼?"
print(f"\n질문 3: {q3}")
result3 = chat_rag(q3)
print(f"답변 3: {result3['answer']}")

print("\n대화형 모드 테스트 성공!")

# 대화 기록 확인
print(f"\n 대화 기록 ({len(chat_history)//2}턴):")
for i in range(0, len(chat_history), 2):
    print(f"\n[턴 {i//2 + 1}]")
    print(f": {chat_history[i].content}")
    print(f": {chat_history[i+1].content[:100]}...")


대화형 RAG 체인 설정...
대화형 RAG 체인 준비 완료!

대화형 테스트:
질문 1: 지원 자격이 뭐야?
답변 1: 지원 자격은 다음과 같이 나와 있습니다:
- 미취업자(고용보험 미가입자)
- 대학 재학생
- 노무제공자
- 개인사업자(부동산임대사업자)
- 개인사업자(부동산임대사업자 이외)
- 개인사업자(고용보험 피보험자)
- 비정규형태근로자
- 결혼이민자등
- 난민
- 법인대표자
- 단체 대표
- 2년제 이상 또는 그와 동등한 수준의 학력
- 산업기사 이상 자격증 보유
- 동일 및 유사계열 경력출신
- 대학(교) 및 학과이며, 기회균형선발 대상인 수급권자, 장애인, 국가유공자 및 그 유족·가족, 중‧장기복무(5년 이상) 제대군인, 한부모가족의 부 또는 모, 전직지원교육 대상자 중 장기복무전역예정군인(10년 이상)기회균형선발 해당자
- 가산점 대상인 자립준비청년(보호종료아동), 결혼이민자등, 난민, 다문화가정의 자녀, 학교밖청소년, 가정밖청소년, 폐업 개인사업자

질문 2: 그럼 제출 서류는 뭐가 필요해?
답변 2: 지원자의 경우 필요한 제출 서류는 다음과 같습니다:
- 입학원서(대학 양식) 1부
- 개인정보이용 동의서 1부
- 졸업(예정)증명서 1부
- 수료(예정) 증명서 1부
- 성적증명서 1부
- 자격증 사본
- 경력증명서
- 근로계약서 또는 확인서
- 사업자등록증명원
- 부가가치세과세표준증명원 또는 신고사실 증명서
- 가산점 관련 서류(난민, 결혼이민자, 자립준비청년, 다문화가정의 자녀, 학교밖청소년, 가정밖청소년, 폐업 개인사업자 등)
- 기회균형선발 서류(수급권자, 장애인, 국가유공자 및 유족‧가족, 중‧장기복무 제대군인, 전직지원교육대상자 중장기복무제대예정군인, 한부모가족 등)

질문 3: 그거 어디에 제출하면 돼?
답변 3: 합격자 선발 관련 제출서류는 한국폴리텍대학 서울강서캠퍼스 교학처(하이테크과정 입시담당자 앞)로 등기우편을 통해 제출하면 됩니다.

대화형 모드 테스트 성공!

 대화 기록 (3턴):

[턴 1]
: 지원 자격이 뭐야?
:

### 참고. 간단 챗봇

In [ ]:
# 직접 질문하기 (출처 포함 버전)
def ask_question_with_source():
    """출처 문서를 포함한 인터랙티브 질문-답변 시스템"""
    print("\n" + "="*60)
    print("PDF RAG 질문-답변 시스템 (출처 포함)")
    print("="*60)
    print("명령어:")
    print("  • 'exit' / 'quit' / '종료' - 시스템 종료")
    print("  • 'detail' / '상세' - 마지막 답변의 상세 정보 보기")
    print("="*60)

    question_count = 0
    last_result = None

    while True:
        question = input("\n질문을 입력하세요: ").strip()

        if question.lower() in ['exit', 'quit', '종료', '나가기']:
            print(f"\n총 {question_count}개의 질문에 답변했습니다.")
            print("시스템을 종료합니다.")
            break

        if question.lower() in ['detail', '상세']:
            if last_result:
                print("\n마지막 답변 상세 정보:")
                print("="*60)
                print(f"질문: {last_result['question']}")
                print(f"\n답변:\n{last_result['answer']}")
                print(f"\n참조 문서 ({len(last_result['source_documents'])}개):")
                for i, doc in enumerate(last_result['source_documents'], 1):
                    page = doc.metadata.get('page', '?')
                    content = doc.page_content[:150].replace('\n', ' ')
                    print(f"\n  [{i}] 페이지 {page}")
                    print(f"      {content}...")
            else:
                print("아직 질문한 내역이 없습니다.")
            continue

        if not question:
            print("질문을 입력해주세요.")
            continue

        try:
            print("검색 및 답변 생성 중...")

            # rag_chain이 출처 포함 버전인 경우
            result = rag_chain.invoke(question)

            question_count += 1
            last_result = {
                'question': question,
                'answer': result['answer'],
                'source_documents': result['source_documents']
            }

            # 결과 출력
            print("\n" + "="*60)
            print("AI 답변:")
            print("="*60)
            print(result['answer'])

            # 참조 문서 간단 표시
            if result['source_documents']:
                sources = []
                for doc in result['source_documents']:
                    page = doc.metadata.get('page', '?')
                    sources.append(f"p.{page}")
                print(f"\n참조: {', '.join(set(sources))}")  # 중복 제거
                print("'detail' 입력 시 상세 정보를 볼 수 있습니다.")

        except Exception as e:
            print(f"오류 발생: {e}")
            print("다시 시도해주세요.")

# 실행
ask_question_with_source()



PDF RAG 질문-답변 시스템 (출처 포함)
명령어:
  • 'exit' / 'quit' / '종료' - 시스템 종료
  • 'detail' / '상세' - 마지막 답변의 상세 정보 보기

질문을 입력하세요: 스마트금융과 어떤거 배워?
검색 및 답변 생성 중...

AI 답변:
빅데이터 분석, 블록체인 개발, 금융데이터 분석, 웹 어플리케이션 개발을 배웁니다. (출처 1, 페이지 8)

참조: p.15, p.8, p.0, p.1
'detail' 입력 시 상세 정보를 볼 수 있습니다.

질문을 입력하세요: q
검색 및 답변 생성 중...

AI 답변:
죄송합니다, 컨텍스트가 제공되지 않았습니다. 해당 질문에 대한 답변을 드릴 수 없습니다.

참조: p.6, p.2, p.11, p.10
'detail' 입력 시 상세 정보를 볼 수 있습니다.

질문을 입력하세요: exit

총 2개의 질문에 답변했습니다.
시스템을 종료합니다.


### 참고 (이후는 패스) GGUF 로컬 모델 설정

In [ ]:
# GGUF 로컬 모델 설정
print("="*60)
print("GGUF 로컬 모델 로딩")
print("="*60)

# Step 1: 패키지 설치 (필요시)
print("필요한 패키지 확인 중...")
try:
    import llama_cpp
    print("llama-cpp-python 이미 설치됨")
except ImportError:
    print("llama-cpp-python 설치 중...")
    !pip install -q llama-cpp-python

# Step 2: 모델 파일 경로 설정
import os

MODEL_PATH = "./pharos_2026_gemma2b.gguf"  # 현재 폴더

# 파일 존재 확인
if os.path.exists(MODEL_PATH):
    file_size = os.path.getsize(MODEL_PATH) / (1024**3)  # GB 단위
    print(f"모델 파일 발견: {MODEL_PATH}")
    print(f"파일 크기: {file_size:.2f} GB")
else:
    print(f"모델 파일을 찾을 수 없습니다: {MODEL_PATH}")
    print("파일 경로를 확인해주세요!")

# Step 3: GGUF 모델 로딩
print("\nGGUF 모델 로딩 중...")
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

# 콜백 설정 (실시간 출력)
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

# GGUF 모델 초기화
llm_gguf = LlamaCpp(
    model_path=MODEL_PATH,
    n_ctx=4096,           # 컨텍스트 윈도우
    n_threads=2,          # CPU 스레드 수 (환경에 맞게 조정)
    n_gpu_layers=0,       # GPU 사용 안 함 (CPU만)
    temperature=0,        # 결정론적 출력
    max_tokens=512,       # 최대 생성 토큰
    top_p=1,
    callback_manager=callback_manager,
    verbose=False
)

print("GGUF 모델 로딩 완료!")

# Step 4: 간단한 테스트
print("\n모델 테스트:")
try:
    test_response = llm_gguf.invoke("안녕하세요")
    print(f"테스트 성공!")
    print(f"응답: {test_response[:100]}...")
except Exception as e:
    print(f"테스트 실패: {e}")


In [ ]:
# GGUF 기반 RAG 체인 구성
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

print("\n GGUF RAG 체인 구성 중...")

# 프롬프트 템플릿 (한국어 최적화)
prompt_gguf = PromptTemplate.from_template(
    """다음 문맥을 참고하여 질문에 한국어로 답변하세요.
문맥에 없는 내용은 모른다고 답변하세요.

문맥:
{context}

질문: {question}

답변:"""
)

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        content = doc.page_content
        page = doc.metadata.get('page', '알 수 없음')
        formatted.append(f"[문서 {i} - 페이지 {page}]\n{content}")
    return "\n\n".join(formatted)

# RAG 체인 구성
rag_chain_gguf = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_gguf
    | llm_gguf
    | StrOutputParser()
)

print("GGUF RAG 체인 생성 완료!")

print("\n체인 구성요소:")
print(f"  LLM: GGUF (pharos_2026_gemma2b)")
print(f"   Retriever: ChromaDB (K={K_RETRIEVAL})")
print(f"   Prompt: 한국어 최적화")
print(f"   실행: CPU 기반")


In [ ]:
# GGUF RAG 체인 테스트
print("\n" + "="*60)
print("GGUF RAG 체인 테스트")
print("="*60)

# 테스트 질문
test_questions = [
    "지원 자격이 어떻게 되나요?",
    "신청 방법을 알려주세요",
    "제출 서류는 무엇인가요?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n[질문 {i}]")
    print(f"{question}")
    print("-" * 60)

    try:
        # 시간 측정
        import time
        start = time.time()

        answer = rag_chain_gguf.invoke(question)

        elapsed = time.time() - start

        print(f"답변: {answer}")
        print(f"소요 시간: {elapsed:.2f}초")

    except Exception as e:
        print(f"에러: {e}")

    print("="*60)
